# HODL-Lec-9-NanoGPT (PyTorch)

In the transformers lesson we used a Transformer **encoder** to label every word of a travel query. In this lesson we flip the problem around: instead of *reading* text, we will build a model that *writes* it.

We will build a small GPT ("Generatively Pre-trained Transformer") from scratch and train it on the complete works of Shakespeare. It is the same architecture as GPT-2/GPT-3, only much smaller. The lesson follows Andrej Karpathy's [nanoGPT](https://github.com/karpathy/nanoGPT) and his video [Let's build GPT: from scratch, in code, spelled out](https://www.youtube.com/watch?v=kCc8FmEb1nY).

**Roadmap**

1. Load the data and turn characters into integers (tokenization)
2. Frame language modeling as "predict the next token"
3. Baseline: a **bigram** model that only looks at the previous character
4. The key idea: **self-attention** as a weighted average over the past
5. Assemble the pieces (multi-head attention, feed-forward, residuals, LayerNorm) into a **GPT**
6. Train it and generate Shakespeare-ish text

## Setup

In [ ]:
import os
import time
import urllib.request

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

# Use the GPU if there is one: CUDA on Linux/Windows, MPS on Apple Silicon
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

## Data: Tiny Shakespeare

The dataset is a single 1MB text file with all of Shakespeare's plays concatenated. We download it once into `data/` (which is gitignored).

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
DATA_PATH = os.path.join("data", "input.txt")

if not os.path.exists(DATA_PATH):
    os.makedirs("data", exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Length of dataset in characters: {len(text):,}")

In [ ]:
print(text[:500])

## Tokenization

A neural network works with numbers, not text, so we need to map text to a sequence of integers (**tokens**).

In the music genre lessons we split text into *words*. Here we go to the other extreme and use individual **characters** as tokens. That gives a tiny vocabulary (65 symbols) and means the model can never see an "out of vocabulary" token, at the price of longer sequences.

> Production GPTs sit in between: they use **sub-word** tokenizers such as Byte-Pair Encoding (BPE). GPT-2's vocabulary has 50,257 tokens, so "Shakespeare" might be 2-3 tokens instead of 11 characters.

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
print("".join(chars))
print(f"vocab_size = {vocab_size}")

In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}   # string -> integer
itos = {i: ch for i, ch in enumerate(chars)}   # integer -> string

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

print(encode("hii there"))
print(decode(encode("hii there")))

Now encode the whole dataset into one long tensor of integers and hold out the last 10% for validation.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"train tokens: {len(train_data):,}   val tokens: {len(val_data):,}")

## Language modeling = next-token prediction

We never feed the whole text to the model at once. Instead we sample random **chunks** of length `block_size` (the *context length*).

A single chunk of `block_size + 1` characters actually contains `block_size` training examples: at every position, the model sees everything up to that point and must predict the next character.

In [ ]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size + 1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print(f"when input is {decode(context.tolist())!r:12} the target is {decode([target.item()])!r}")

Training on contexts of every length from 1 to `block_size` is deliberate: at generation time the model has to start from very little context.

We also stack several chunks into a **batch** so the GPU has independent work to do in parallel.

In [ ]:
def get_batch(split, batch_size, block_size):
    """Sample a batch of random chunks. y is x shifted one character to the left."""
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in ix])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train", batch_size=4, block_size=8)
print("inputs:", xb.shape)
print(xb)
print("targets:", yb.shape)
print(yb)

## Baseline: the bigram model

The simplest possible language model predicts the next character from **only the current character**. It is just a lookup table: row `i` of an embedding matrix holds the logits (unnormalized scores) for which character follows character `i`.

A sanity check before training: with 65 characters a model that guesses uniformly should have a loss of $-\ln(1/65) \approx 4.17$.

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, vocab_size)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]                       # only the last time step
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


bigram = BigramLanguageModel(vocab_size).to(device)
logits, loss = bigram(xb, yb)
print(logits.shape)
print(f"initial loss: {loss.item():.4f}  (expected ~{torch.log(torch.tensor(vocab_size)).item():.4f})")

The initial loss is noticeably *higher* than 4.17. `nn.Embedding` initializes its weights from N(0, 1), so the untrained model already makes confident (and wrong) guesses. A well-initialized model starts close to uniform; we will use a smaller init for the GPT below and see it start right at ~4.2.

Generation starts from a single newline character (token 0). Untrained, the output is random noise:

In [ ]:
start = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(bigram.generate(start, max_new_tokens=200)[0].tolist()))

### A reusable training loop

Language models are usually trained for a number of **steps** (iterations) rather than epochs. Because a single batch gives a noisy loss, `estimate_loss` averages the loss over many batches for both splits.

In [ ]:
@torch.no_grad()
def estimate_loss(model, batch_size, block_size, eval_iters=100):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size, block_size)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def train(model, max_iters, batch_size, block_size, lr, eval_interval):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    history = {"step": [], "train": [], "val": []}
    t0 = time.time()
    for step in range(max_iters + 1):
        if step % eval_interval == 0:
            losses = estimate_loss(model, batch_size, block_size)
            history["step"].append(step)
            history["train"].append(losses["train"])
            history["val"].append(losses["val"])
            print(f"step {step:5d}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}"
                  f"  ({time.time() - t0:.0f}s)")
        if step == max_iters:
            break
        xb, yb = get_batch("train", batch_size, block_size)
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return history

In [ ]:
bigram_history = train(bigram, max_iters=3000, batch_size=32, block_size=8, lr=1e-2, eval_interval=500)

In [ ]:
print(decode(bigram.generate(start, max_new_tokens=300)[0].tolist()))

Better: it has learned which characters tend to follow which, so we see word-like chunks and line breaks. But the loss plateaus around 2.5 because **the tokens don't talk to each other**: every prediction ignores everything except the immediately preceding character.

To do better, each position needs to gather information from the positions before it. That is exactly what self-attention does.

## The mathematical trick behind self-attention

Take a toy batch: `B` sequences, `T` time steps, `C` channels per token. We want position `t` to receive a summary of positions `0..t` (never the future, since that is what we are trying to predict).

The simplest summary is the **average** of all previous token vectors. Version 1 uses an explicit loop:

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

# version 1: for loops
xbow = torch.zeros((B, T, C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t + 1]            # (t+1, C)
        xbow[b, t] = torch.mean(xprev, 0)

Loops are slow. **Version 2:** the same thing as a single matrix multiply with a lower-triangular weight matrix whose rows sum to 1:

In [ ]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
print(wei)
xbow2 = wei @ x     # (T, T) @ (B, T, C) -> (B, T, C)
print("matches version 1:", torch.allclose(xbow, xbow2, atol=1e-6))

**Version 3:** build the same weights with `softmax`. Start from zeros (all positions equally interesting), set the future to $-\infty$ so it gets zero weight, then normalize with softmax.

This is the form we want, because the zeros can be replaced by **data-dependent affinities**: how interesting token `j` is to token `t`.

In [ ]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
print("matches version 1:", torch.allclose(xbow, xbow3, atol=1e-6))

### Version 4: self-attention

Each token emits three vectors via learned linear layers:

- **query** — "what am I looking for?"
- **key** — "what do I contain?"
- **value** — "if you attend to me, here is what I'll give you"

The affinity between positions `t` and `j` is the dot product `q_t · k_j`. High affinity means token `t` pulls in a lot of token `j`'s value.

In [ ]:
torch.manual_seed(1337)
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)    # (B, T, 16)
q = query(x)  # (B, T, 16)

wei = q @ k.transpose(-2, -1) * head_size ** -0.5   # (B, T, 16) @ (B, 16, T) -> (B, T, T)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
print(out.shape)
print(wei[0].detach().round(decimals=2))

Notice the weights are no longer uniform: each row is a different, data-dependent mix of the past.

A few notes worth remembering:

- **Attention is a communication mechanism.** Tokens are nodes in a graph, each aggregating a weighted sum from the nodes that point to it. It has no built-in notion of order, which is why we add **position embeddings**.
- **Batches don't talk to each other.** Each of the `B` sequences is processed independently.
- **Encoder vs decoder.** The triangular mask is what makes this a *decoder* block. Delete the `masked_fill` line and every token sees every other token, which is the *encoder* we used for slot filling in lesson 7.
- **Self vs cross attention.** Here queries, keys and values all come from `x`. In cross-attention (e.g. translation), keys and values come from a different source.
- **Scaled attention.** We multiply by `1/sqrt(head_size)`. Without it, the variance of `q·k` grows with `head_size`, softmax saturates towards one-hot, and each token ends up attending to a single other token.

In [ ]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)
print("unscaled var:", (q @ k.transpose(-2, -1)).var().item())
print("scaled var:  ", (q @ k.transpose(-2, -1) * head_size ** -0.5).var().item())

logits = torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])
print("softmax(x):   ", F.softmax(logits, dim=-1).round(decimals=3))
print("softmax(8*x): ", F.softmax(logits * 8, dim=-1).round(decimals=3), "<- nearly one-hot")

## Building blocks of a GPT

Now we package these ideas into modules. They are the "lego blocks" of the Transformer from lesson 7, rebuilt by hand so that nothing is hidden.

In [ ]:
# Hyperparameters. Sized to train in a few minutes on a laptop GPU (Apple MPS) or Colab T4.
# On CPU, halve n_embd and n_layer and set max_iters = 1000.
batch_size = 64
block_size = 128     # context length (characters)
n_embd = 192         # embedding dimension
n_head = 6           # head_size = n_embd // n_head = 32
n_layer = 6          # number of Transformer blocks
dropout = 0.2
learning_rate = 1e-3
max_iters = 3000
eval_interval = 300

**One head of self-attention.** This is version 4 from above as an `nn.Module`. The causal mask is stored as a *buffer* (state that moves with `.to(device)` but is not trained). We also keep the latest attention weights so we can plot them later.

In [ ]:
class Head(nn.Module):
    """One head of causal self-attention."""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
        self.last_attention = None

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, hs)
        q = self.query(x)  # (B, T, hs)
        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5          # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        self.last_attention = wei.detach()
        wei = self.dropout(wei)
        v = self.value(x)  # (B, T, hs)
        return wei @ v     # (B, T, hs)

**Multi-head attention.** Several heads run in parallel, each free to look for a different kind of relationship (e.g. "the previous vowel", "the start of this word"). Their outputs are concatenated and mixed by a projection layer.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

**Feed-forward.** Attention is the *communication* step; this is the *computation* step, applied to each token independently. The hidden layer is 4x wider, as in the original Transformer paper.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

**Transformer block.** Communication followed by computation, with two tricks that make deep networks trainable:

- **Residual connections** (`x = x + ...`): gradients flow straight down the "residual highway", and each block only learns a small correction.
- **LayerNorm**: normalizes each token's features. GPT-2 applies it *before* each sub-layer ("pre-norm"), a small change from the original paper that trains more stably.

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## The full GPT

1. Look up a **token embedding** and a **position embedding** for each input position and add them.
2. Pass through `n_layer` Transformer blocks.
3. Final LayerNorm, then a linear layer (the "language model head") produces logits over the vocabulary at **every** position.

Generation is the same as for the bigram model, with one change: we crop the context to the last `block_size` tokens because the position embedding table only has `block_size` rows. We also add two common sampling controls:

- **temperature**: divide the logits before softmax. `< 1` is more conservative, `> 1` more adventurous.
- **top-k**: only sample from the `k` most likely characters.

In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        # Small normal init (as in GPT-2) instead of PyTorch's defaults
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                                 # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, C)
        x = tok_emb + pos_emb                                                     # (B, T, C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                                  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        self.train()
        return idx


torch.manual_seed(1337)
model = GPTLanguageModel().to(device)
num_params = sum(p.numel() for p in model.parameters())
print(f"{num_params / 1e6:.2f}M parameters")

For scale: GPT-2 small has 124M parameters (12 layers, 768-dim embeddings, 1024-token context) and GPT-3 has 175B. Our model is the same recipe, roughly 50x smaller than GPT-2 small.

**OPTIONAL**: check the parameter count by hand. Each block has 4 attention matrices of size `n_embd x n_embd` (Q, K, V and the output projection), two feed-forward layers of size `n_embd x 4·n_embd`, plus biases and LayerNorms.

In [ ]:
per_block = (
    3 * n_embd * n_embd                       # Q, K, V (no bias)
    + n_embd * n_embd + n_embd                # attention output projection
    + n_embd * 4 * n_embd + 4 * n_embd        # FFN up
    + 4 * n_embd * n_embd + n_embd            # FFN down
    + 2 * 2 * n_embd                          # two LayerNorms (weight + bias)
)
embeddings = vocab_size * n_embd + block_size * n_embd
head = 2 * n_embd + n_embd * vocab_size + vocab_size   # final LayerNorm + lm_head
print(f"by hand: {n_layer * per_block + embeddings + head:,}   pytorch: {num_params:,}")

## Train the model

Same loop as the bigram model. Expect a few minutes on a laptop GPU. The validation loss should drop from ~4.2 to around 1.5.

In [ ]:
history = train(model, max_iters=max_iters, batch_size=batch_size, block_size=block_size,
                lr=learning_rate, eval_interval=eval_interval)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["step"], history["train"], "o-", label="train loss", markersize=4)
plt.plot(history["step"], history["val"], "o-", label="val loss", markersize=4)
plt.axhline(bigram_history["val"][-1], color="gray", linestyle="--", label="bigram val loss")
plt.xlabel("step")
plt.ylabel("cross-entropy loss")
plt.title("GPT training curves")
plt.legend()
plt.show()

## Generate Shakespeare

Let's see what the model writes, starting from a newline.

In [ ]:
torch.manual_seed(42)
print(decode(model.generate(start, max_new_tokens=500)[0].tolist()))

Not real Shakespeare, but it has picked up character names in capitals, speaker/line structure, punctuation, and many real English words, all from raw characters.

We can also give it a **prompt** and try the sampling controls:

In [ ]:
prompt = "ROMEO:\nBut soft, "
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

for temperature in [0.5, 1.0, 1.5]:
    torch.manual_seed(42)
    out = model.generate(context, max_new_tokens=200, temperature=temperature, top_k=20)
    print(f"--- temperature = {temperature} ---")
    print(decode(out[0].tolist()))
    print()

## Peek inside: what does a head attend to?

Each row of the heatmap below is one position in the prompt; the columns show how much it attends to every earlier position. The upper triangle is empty because of the causal mask. Different heads in different layers learn different patterns.

In [ ]:
sample = "First Citizen:\nBefore we proceed any further, hear me speak."
idx = torch.tensor([encode(sample)], dtype=torch.long, device=device)
model.eval()
with torch.no_grad():
    model(idx)
model.train()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (layer, head) in zip(axes, [(0, 0), (n_layer // 2, 0), (n_layer - 1, 0)]):
    att = model.blocks[layer].sa.heads[head].last_attention[0].cpu()
    ax.imshow(att, cmap="viridis")
    ax.set_title(f"layer {layer}, head {head}")
    labels = [repr(c)[1:-1] for c in sample]
    ax.set_xticks(range(len(sample)))
    ax.set_xticklabels(labels, fontsize=6)
    ax.set_yticks(range(len(sample)))
    ax.set_yticklabels(labels, fontsize=6)
plt.tight_layout()
plt.show()

## From nanoGPT to ChatGPT

What we built is the **pre-training** stage: a decoder-only Transformer that has learned to continue text in the style of its training data. It answers nothing and follows no instructions; it just *babbles Shakespeare*.

Getting from here to an assistant like ChatGPT takes two more ingredients:

| Stage | Our model | GPT-3 / ChatGPT |
|---|---|---|
| **Tokens** | 65 characters | ~50k-100k BPE sub-words |
| **Data** | 1MB of Shakespeare | Hundreds of billions of tokens of internet text |
| **Model** | ~2.7M params, 6 layers | 175B params, 96 layers |
| **Fine-tuning** | none | Supervised fine-tuning on prompt/answer pairs, then RLHF to align with human preferences |

The architecture barely changes. Scale, data and post-training do the rest. Later lessons pick up from here: lesson 10 (RAG) feeds external knowledge into an LLM's context, and lesson 10.5 fine-tunes a pre-trained LLM cheaply with LoRA.

## Exercises

1. **Scale it.** Increase `n_layer`, `n_embd` and `block_size`. How low can you push the validation loss before it starts to overfit (train loss keeps dropping, val loss rises)?
2. **Drop the mask.** Remove the `masked_fill` line in `Head`. The training loss collapses towards zero. Why is that cheating, and what does the generated text look like?
3. **Use PyTorch's fused attention.** Replace the body of `Head.forward` with `F.scaled_dot_product_attention(q, k, v, is_causal=True)`. Is it faster?
4. **Learning-rate schedule.** Add linear warmup + cosine decay (as in lessons 2-3) and compare curves.
5. **Weight tying.** GPT-2 shares weights between `token_embedding_table` and `lm_head` (`self.lm_head.weight = self.token_embedding_table.weight`). How many parameters does that save, and does the loss change?
6. **Your own corpus.** Swap `input.txt` for a different text file (song lyrics from lesson 5, your own writing, source code...).